In [2]:
# !pip install --upgrade transformers torch
#pip install --upgrade transformers huggingface_hub
from transformers import pipeline
import re

from mongo_client import mongo_client, articles_collection
from datetime import datetime, timezone
from functions import combine_paragraphs

from dotenv import load_dotenv
load_dotenv()

try:
    mongo_client.admin.command("ping")
    print("✅ Connected to MongoDB Atlas!")
except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    raise

✅ Connected to MongoDB Atlas!


In [3]:
def clean_text(text: str) -> str:
    """
    Minimal normalization:
      • lowercase everything
      • strip URLs
      • collapse multiple spaces / new-lines
      • keep only letters, numbers, punctuation . , : ; - and spaces
    """
    text = text.lower()
    # remove urls
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    # keep selected characters, drop the rest
    text = re.sub(r"[^a-z0-9.,:;\-\s]", " ", text)
    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
        # {"meta.date": {
        #     "$gt": datetime(2024,1,11),
        #     "$lt": datetime(2024,12,31),
        #     },

In [4]:
limit = 2

articles_to_process = list(
    articles_collection.find({
        "title": "The Hydrogen Stream: US team produces hydrogen from ocean water",    
        }, 
        {"_id": 1, 
         "paragraphs": 1, 
         "nodes":1}
    )
    .limit(limit)    # Limit the number of articles
)

In [81]:
articles_to_process

[{'_id': ObjectId('67fa626038fb90ab92f8c1a1'),
  'paragraphs': [{'p1': 'A team of US researchers engineered a double-membrane system to minimize chloride oxidizing the anode. Meanwhile, the Spanish government supports Destinus to develop its hydrogen-powered supersonic plane and test it in 2024.',
    'p2': "A representation of the team's bipolar membrane system that converts seawater into hydrogen gas",
    'p3': 'Image: Nina Fujikawa/SLAC National Accelerator Laboratory',
    'p4': 'ASLAC-Stanford teamhas produced hydrogen directly from ocean water, funneling it through a double-membrane system. “Many water-to-hydrogen systems today try to use a monolayer or single-layer membrane. Our study brought two layers together,”saidAdam Nielander, a researcher at the SLAC-Stanford joint institute. The system mitigates chloride from getting to the anode and oxidizing it. Protons pass through one of the membrane layers to a place where they can be collected and turned into hydrogen gas by inter

In [5]:
# a first prompt to split between deployment vs manufacturing? 

labels = ["battery", "biomass", "geothermal", "hydroelectric", "hydrogen", "nuclear", "solar", "steel", "vehicle", "wind"]

clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

for article in articles_to_process[1:]:
    text = combine_paragraphs(article)
    product_nodes = [node for node in article["nodes"] if node.get("type") == "product"]

    for product_node in product_nodes:
        product_name = product_node.get("name")
        #establish premise
        premise = f"Item: {product_name}. Context: {text}"
        result = clf(premise, labels) 
        print(f"{product_name}: labelled as - {result['labels'][0]}")

Device set to use mps:0


hydrogen-powered supersonic plane: labelled as - hydrogen
green hydrogen: labelled as - hydrogen
green ammonia: labelled as - hydrogen


In [7]:
clean_text(text)

'a team of us researchers engineered a double-membrane system to minimize chloride oxidizing the anode. meanwhile, the spanish government supports destinus to develop its hydrogen-powered supersonic plane and test it in 2024. this content is protected by copyright and may not be reused. if you want to cooperate with us and would like to reuse some of our content, please contact:editors pv-magazine.com. more articles from sergio matalucci ocean water to hydrogen will change all economics of hydrogen production. technology to economize and scale it for mass production and bulk production would be the race for next decade. manufacturing portable hydrogen plants will permanently change the world. rain water harvesting and recycling it for hydrogen production and then using it in hydrogen fuel cell to produce power will solve electrification issues while addressing drinking water solutions in remote hilly villages in asian and european subcontinent. please be mindful of ourcommunity standar

In [28]:
result['labels'][0]

'vehicle'